# Time series analysis

In [ ]:
# ============================================================
# 1. SETUP
# ============================================================

import os
import re
import sys
import json
import math
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer, OneHotEncoder
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import scipy.sparse as sp

warnings.filterwarnings("ignore")

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)

In [ ]:
### Creating comorbiditiy

# ============================================================
# RULE-BASED COMORBIDITY INDICATORS (UPDATED + NEW CATEGORIES)
# - Adds endocrine/thyroid + musculoskeletal + mental health +
#   GI + neuro_headache (migraine) + lymphatic/edema (optional)
# - Treats "none/no/unknown/NKA/NKDA" etc. as missing text
# - Saves outputs to CSV by default (no pyarrow needed)
#   (Parquet is optional if pyarrow/fastparquet is installed)
# ============================================================

import os
import re
import numpy as np
import pandas as pd

OUTDIR = "/Users/ariellerothman/Desktop/Capstone/Outputs"
os.makedirs(OUTDIR, exist_ok=True)

# Update this to your dataframe name if needed:
# df_clean = <your cleaned case-level dataframe>

FREE_TEXT_FIELDS = ["HISTORY", "CUR_ILL", "ALLERGIES", "PRIOR_VAX", "LAB_DATA", "OTHER_MEDS"]

# Terms that should be treated as "no information" (so they become missing)
NULL_TOKENS = {
    "none", "no", "n", "na", "n/a", "unknown", "unk", "none known", "not known",
    "nka", "nkda", "no known allergies", "no known drug allergies",
    "denies", "denies allergies", "no allergies", "nil", "negative"
}

_keep = re.compile(r"[^a-z0-9\s\-\/]")
_ws = re.compile(r"\s+")

def normalize_text(x) -> str:
    """Normalize free-text: lowercase, remove urls/emails/specials, collapse whitespace,
    and convert null-like tokens to empty string."""
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    if not x:
        return ""

    # Common "null-like" entries
    if x in NULL_TOKENS:
        return ""

    # remove urls/emails
    x = re.sub(r"http\S+|www\S+|https\S+", " ", x)
    x = re.sub(r"\S+@\S+", " ", x)

    # keep alnum + spaces + hyphen + slash
    x = _keep.sub(" ", x)
    x = _ws.sub(" ", x).strip()

    # If after normalization it's basically a null token, drop it
    if x in NULL_TOKENS:
        return ""

    return x

# ---------- UPDATED CATEGORY MAP ----------
# Notes:
# - Add categories where your "other" bucket showed lots of valid clinical signal:
#   hypothyroidism/thyroid, arthritis/OA, migraine, depression/anxiety, GERD, edema/lymphedema.
# - Keep patterns conservative using word boundaries (\b) to reduce false positives.
CATEGORY_MAP = {
    "CUR_ILL": {
        "acute_infection": [
            r"\bcovid\b", r"\bsars[-\s]?cov[-\s]?2\b", r"\binfluenza\b", r"\bflu\b",
            r"\binfection\b", r"\bpneumonia\b", r"\bbronchitis\b", r"\bviral\b", r"\bbacterial\b"
        ],
        "chronic_cardiovascular": [
            r"\bhypertension\b", r"\bhtn\b", r"\bcad\b", r"\bcoronary\b", r"\bafib\b",
            r"\bheart failure\b", r"\bchf\b", r"\barrhythmia\b", r"\bmi\b", r"\bmyocardial infarction\b"
        ],
        "chronic_respiratory": [
            r"\basthma\b", r"\bcopd\b", r"\bemphysema\b", r"\bchronic obstructive\b",
            r"\bsleep apnea\b", r"\bosa\b"
        ],
        "metabolic_endocrine": [
            r"\bdiabetes\b", r"\bdm\b", r"\bdm2\b", r"\btype\s?2\b", r"\bobesity\b",
            r"\bhyperlipid\w*\b", r"\bdyslipid\w*\b"
        ],
        # NEW: thyroid/endocrine explicitly (often shows up in your OTHER examples)
        "endocrine_thyroid": [
            r"\bthyroid\b", r"\bhypothyroid\w*\b", r"\bhyperthyroid\w*\b",
            r"\bhashimoto\w*\b", r"\bgraves\b"
        ],
        "autoimmune_inflammatory": [
            r"\bautoimmune\b", r"\blupus\b", r"\bsle\b", r"\bra\b", r"\brheumatoid arthritis\b",
            r"\bmultiple sclerosis\b", r"\bms\b", r"\bibs\b", r"\bibd\b", r"\bceliac\b"
        ],
        "neurologic": [
            r"\bseizure\b", r"\bepilepsy\b", r"\bstroke\b", r"\btia\b", r"\bparkinson\w*\b",
            r"\bneuropath\w*\b", r"\bdementia\b", r"\balzheimer\w*\b"
        ],
        # NEW: migraine/headache bucket (showed up in your examples)
        "neuro_headache": [
            r"\bmigraine\w*\b", r"\bheadache\w*\b", r"\bcluster headache\b"
        ],
        "cancer_immunocompromised": [
            r"\bcancer\b", r"\bmalignan\w*\b", r"\bchemotherap\w*\b", r"\bradiation\b",
            r"\btransplant\b", r"\bhiv\b", r"\baids\b",
            r"\bimmunosupp\w*\b", r"\bimmunocompromis\w*\b"
        ],
        # NEW (optional): edema/lymphedema bucket (shows in your examples)
        "lymphatic_edema": [
            r"\blymph(edema|oedema)\b", r"\bedema\b", r"\bswelling\b"
        ],
    },

    "HISTORY": {
        "cardiovascular": [
            r"\bhtn\b", r"\bhypertension\b", r"\bcad\b", r"\bcoronary\b", r"\bafib\b",
            r"\bheart failure\b", r"\bchf\b", r"\bmi\b", r"\bmyocardial infarction\b",
            r"\bvalve\b", r"\baortic stenosis\b"
        ],
        "respiratory": [
            r"\basthma\b", r"\bcopd\b", r"\bemphysema\b", r"\bchronic obstructive\b",
            r"\bosa\b", r"\bsleep apnea\b"
        ],
        "metabolic": [
            r"\bdiabetes\b", r"\bdm\b", r"\bdm2\b", r"\bobesity\b",
            r"\bhyperlipid\w*\b", r"\bdyslipid\w*\b"
        ],
        # NEW: thyroid/endocrine (common in HISTORY OTHER)
        "endocrine_thyroid": [
            r"\bthyroid\b", r"\bhypothyroid\w*\b", r"\bhyperthyroid\w*\b",
            r"\bhashimoto\w*\b", r"\bgraves\b"
        ],
        # NEW: musculoskeletal (arthritis/OA showed in your OTHER samples)
        "musculoskeletal": [
            r"\barthritis\b", r"\boveruse\b", r"\bosteoarthritis\b", r"\boa\b",
            r"\bgout\b", r"\bfibromyalgia\b", r"\bchronic pain\b"
        ],
        # NEW: mental health (often appears in HISTORY strings)
        "mental_health": [
            r"\bdepress\w*\b", r"\banxiety\b", r"\bptsd\b", r"\bbipolar\b",
            r"\bschizo\w*\b"
        ],
        # NEW: GI (GERD appears a lot)
        "gastrointestinal": [
            r"\bgerd\b", r"\bacid reflux\b", r"\bgastroesophageal\b",
            r"\bibs\b", r"\bconstipat\w*\b", r"\bdiverticul\w*\b"
        ],
        "kidney": [
            r"\bckd\b", r"\bchronic kidney\b", r"\brenal\b", r"\bdialysis\b",
            r"\brenal failure\b", r"\bkidney transplant\b", r"\btransplant\b"
        ],
        "clotting": [
            r"\bdvt\b", r"\bpe\b", r"\bthromb\w*\b", r"\bclot\b",
            r"\bembol\w*\b", r"\bantiphospholipid\b"
        ],
        "autoimmune": [
            r"\bautoimmune\b", r"\blupus\b", r"\bsle\b", r"\bra\b",
            r"\bms\b", r"\bmultiple sclerosis\b"
        ],
        "cancer_immunocompromised": [
            r"\bcancer\b", r"\bmalignan\w*\b", r"\bchemotherap\w*\b", r"\bradiation\b",
            r"\btransplant\b", r"\bhiv\b", r"\baids\b", r"\bimmunosupp\w*\b"
        ],
        "neurologic": [
            r"\bstroke\b", r"\btia\b", r"\bseizure\b", r"\bepilepsy\b",
            r"\bmigraine\w*\b", r"\bneuropath\w*\b", r"\bparkinson\w*\b"
        ],
        # NEW (optional): lymphatic/edema
        "lymphatic_edema": [
            r"\blymph(edema|oedema)\b", r"\bedema\b"
        ],
    },

    "ALLERGIES": {
        "drug_allergy": [
            r"\bpenicillin\b", r"\bsulfa\b", r"\bnsaid\b", r"\baspirin\b",
            r"\bcephalosporin\b", r"\bceclor\b", r"\bkeflex\b", r"\bdoxycycline\b"
        ],
        "food_allergy": [
            r"\bpeanut\b", r"\bnut\b", r"\begg\b", r"\bmilk\b", r"\bshellfish\b", r"\bgarlic\b"
        ],
        "latex_other_contact": [
            r"\blatex\b", r"\bdetergent\b", r"\bfragrance\b"
        ],
        "anaphylaxis_history": [
            r"\banaphylax\w*\b", r"\bepi[-\s]?pen\b", r"\bepinephrine\b"
        ],
    },

    "PRIOR_VAX": {
        "prior_covid_vax": [r"\bcovid\b", r"\bpfizer\b", r"\bmoderna\b", r"\bjanssen\b", r"\bnovavax\b"],
        "prior_flu_vax": [r"\bflu\b", r"\binfluenza\b"],
        # broaden common vaccines
        "prior_other_vax": [
            r"\bshingrix\b", r"\bshingles\b", r"\btetanus\b", r"\brabies\b",
            r"\bh1n1\b", r"\bmmr\b", r"\bhepatitis\b", r"\bvaricella\b"
        ],
        "prior_vax_reaction": [r"\breaction\b", r"\ballergic\b", r"\banaphylax\w*\b", r"\bside effect\b", r"\bsyncope\b"],
    },

    "LAB_DATA": {
        "cardiac_markers": [r"\btroponin\b", r"\bbnp\b", r"\bck[-\s]?mb\b"],
        "coagulation": [r"\bd[-\s]?dimer\b", r"\binr\b", r"\bptt\b", r"\bfibrinogen\b"],
        "inflammation": [r"\bcrp\b", r"\besr\b", r"\bferritin\b"],
        "cbc": [r"\bwbc\b", r"\bplatelet\w*\b", r"\bhemoglobin\b", r"\bhgb\b"],
        # optional: vitals/procedures bucket if you want to reduce LAB_DATA__other size
        "vitals_procedures": [r"\bbp\b", r"\bhr\b", r"\bspo2\b", r"\bultrasound\b", r"\bct\b", r"\bmri\b", r"\bx[-\s]?ray\b"],
    },

    "OTHER_MEDS": {
        "anticoagulant": [
            r"\bwarfarin\b", r"\bheparin\b", r"\beliquis\b", r"\bxarelto\b",
            r"\bapixaban\b", r"\brivaroxaban\b"
        ],
        "statin": [r"\bstatin\b", r"\batorvastatin\b", r"\bsimvastatin\b", r"\brosuvastatin\b"],
        "immunosuppressant": [r"\bprednisone\b", r"\bsteroid\b", r"\bmethotrexate\b", r"\btacrolimus\b", r"\bcyclosporine\b"],
        "diabetes_meds": [r"\binsulin\b", r"\bmetformin\b", r"\bsemaglutide\b", r"\bozempic\b", r"\bglp[-\s]?1\b"],
        # NEW: thyroid meds commonly present in meds lists
        "thyroid_meds": [r"\blevothyroxine\b", r"\bsynthroid\b", r"\bliothyronine\b"],
        # NEW: antidepressants/anxiolytics common in VAERS meds lists
        "psych_meds": [r"\bsertraline\b", r"\bfluoxetine\b", r"\bcitalopram\b", r"\bescitalopram\b", r"\bbuspirone\b", r"\balprazolam\b"],
    },
}

def compile_map(category_map):
    compiled = {}
    for col, cats in category_map.items():
        compiled[col] = {cat: [re.compile(p) for p in pats] for cat, pats in cats.items()}
    return compiled

COMPILED_MAP = compile_map(CATEGORY_MAP)

def flag_any(text: str, patterns) -> int:
    if not text:
        return 0
    return int(any(p.search(text) for p in patterns))

# ----------------------------
# APPLY TO df_clean
# ----------------------------
df_clean = df_clean.copy()

for col in FREE_TEXT_FIELDS:
    if col not in df_clean.columns:
        continue

    norm_col = f"{col}_NORM"
    df_clean[norm_col] = df_clean[col].apply(normalize_text)

    # missingness indicator after null-token normalization
    df_clean[f"{col}_MISSING"] = (df_clean[norm_col].str.len() == 0).astype("int8")

    # dictionary flags
    if col in COMPILED_MAP:
        cat_cols = []
        for cat, pats in COMPILED_MAP[col].items():
            outcol = f"{col}__{cat}"
            df_clean[outcol] = df_clean[norm_col].apply(lambda t: flag_any(t, pats)).astype("int8")
            cat_cols.append(outcol)

        # "other" = has text but did not match any category
        df_clean[f"{col}__other"] = (
            (df_clean[f"{col}_MISSING"] == 0) & (df_clean[cat_cols].sum(axis=1) == 0)
        ).astype("int8")

# ----------------------------
# SAVE ENGINEERED FEATURES
# ----------------------------
engineered_cols = [c for c in df_clean.columns if "__" in c or c.endswith("_MISSING")]

comorb_df = df_clean[["VAERS_ID"] + engineered_cols].copy()

# 1) Always save CSV (no extra deps)
csv_path = os.path.join(OUTDIR, "comorbidity_indicators.csv")
comorb_df.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path, "shape:", comorb_df.shape)

# 2) Optional Parquet (only if engine installed)
parquet_path = os.path.join(OUTDIR, "comorbidity_indicators.parquet")
try:
    comorb_df.to_parquet(parquet_path, index=False)
    print("Saved Parquet:", parquet_path)
except Exception as e:
    print("Parquet not saved (missing engine). CSV is saved and sufficient.")
    print("Parquet error:", repr(e))

# ----------------------------
# QUICK DIAGNOSTICS
# ----------------------------
print("\nIndicator count:", len(engineered_cols))
top_counts = comorb_df[engineered_cols].sum().sort_values(ascending=False).head(30)
diag_df = pd.DataFrame({"count": top_counts, "percent": (top_counts / len(comorb_df) * 100).round(3)})
print("\nTop indicators by prevalence:")
print(diag_df)

# Optional: show "OTHER" examples for each field to see what you're missing
def show_other_examples(field, n=10):
    other_col = f"{field}__other"
    if other_col not in df_clean.columns:
        return
    mask = (df_clean[other_col] == 1) & (df_clean[field].notna())
    print(f"\n{field} OTHER examples (n={n}):")
    print(df_clean.loc[mask, field].head(n))

for fld in ["HISTORY", "CUR_ILL", "ALLERGIES", "PRIOR_VAX", "LAB_DATA", "OTHER_MEDS"]:
    if fld in df_clean.columns:
        show_other_examples(fld, n=10)

In [ ]:
# ============================================================
# 17. SYMPTOM TEXT PREPROCESSING
#     Updated supervised leakage cleanup with additional
#     quasi-administrative terms based on current model output
# ============================================================

import re
import pandas as pd

# --------------------------------------------------
# Base text cleaning
# --------------------------------------------------
_url_re = re.compile(r"http\S+|www\S+|https\S+")
_email_re = re.compile(r"\S+@\S+")
_num_re = re.compile(r"\d+")
_keep_re = re.compile(r"[^a-z0-9\s\-]")
_ws2 = re.compile(r"\s+")

def clean_symptom_text(x):
    if pd.isna(x) or x == "":
        return ""
    x = str(x).strip().lower()
    x = _url_re.sub(" ", x)
    x = _email_re.sub(" ", x)
    x = _num_re.sub(" __NUM__ ", x)
    x = _keep_re.sub(" ", x)
    x = _ws2.sub(" ", x).strip()
    return x

df_clean["SYMPTOM_TEXT_CLEAN"] = df_clean["SYMPTOM_TEXT"].apply(clean_symptom_text)

# --------------------------------------------------
# Strong supervised leakage scrubbing for severity prediction
# Removes direct outcome / adjudication / downstream-care terms
# plus residual administrative/report-style phrases
# --------------------------------------------------
LEAKAGE_PATTERNS = [re.compile(p) for p in [

    # ----------------------------------------------
    # Hospital / admission / downstream care
    # ----------------------------------------------
    r"\bhospital\w*\b",
    r"\badmit\w*\b",
    r"\binpatient\b",
    r"\bdischarg\w*\b",
    r"\bed\b",
    r"\ber\b",
    r"\bemergency\b",
    r"\bemergency room\b",
    r"\burgent\b",
    r"\burgent care\b",
    r"\btransport\w*\b",
    r"\btransfer\w*\b",
    r"\bambulance\b",
    r"\bems\b",
    r"\bhospice\b",

    # ----------------------------------------------
    # Death / life-threatening / rescue care
    # ----------------------------------------------
    r"\bdeath\b",
    r"\bdied\b",
    r"\bdead\b",
    r"\bfatal\b",
    r"\bpassed away\b",
    r"\bautopsy\b",
    r"\blife[-\s]?threaten\w*\b",
    r"\bintubat\w*\b",
    r"\bventilat\w*\b",
    r"\bicu\b",
    r"\barrest\b",

    # ----------------------------------------------
    # Seriousness / adjudication labels
    # ----------------------------------------------
    r"\bserious\w*\b",
    r"\bnon[-\s]?serious\w*\b",
    r"\bmedically significant\b",
    r"\bmedically important\b",
    r"\bcriterion\b",
    r"\bcriteria\b",
    r"\boutcome\b",
    r"\bdisability\b",
    r"\bpermanent\b",
    r"\bprolonged\b",

    # ----------------------------------------------
    # Administrative / report-template phrasing
    # ----------------------------------------------
    r"\breported cause\b",
    r"\breport medically\b",
    r"\bcriterion hospitalization\b",
    r"\bhospitalization medically\b",
    r"\bsignificant outcome\b",
    r"\bnarrative\b",
    r"\breport\b",
    r"\breported\b",
    r"\bspontaneous\b",
    r"\bcase\b",
    r"\bsource\b",
    r"\bpatient unknown\b",
    r"\bunknown\b",

    # ----------------------------------------------
    # Residual report-style / quasi-administrative phrases
    # seen in current model outputs
    # ----------------------------------------------
    r"\bevent resulted\b",
    r"\bevents resulted\b",
    r"\bresulted\b",
    r"\bcaused prolonged\b",
    r"\bmedical center\b",
    r"\bclinic care\b",
    r"\bwent care\b",
    r"\bvisited\b",
    r"\bperformed\b",

    # ----------------------------------------------
    # Other report-level phrasing seen previously
    # ----------------------------------------------
    r"\broom\b",
    r"\bdepartment\b",
    r"\bvisit\b",
    r"\bwent room\b",
    r"\bwent doctor\b",
    r"\bdoctor visit\b"
]]

def scrub_leakage_terms_strong(x):
    if pd.isna(x) or x == "":
        return ""
    x = str(x)
    for p in LEAKAGE_PATTERNS:
        x = p.sub(" ", x)
    x = _ws2.sub(" ", x).strip()
    return x

df_clean["SYMPTOM_TEXT_CLEAN_NOLEAK"] = (
    df_clean["SYMPTOM_TEXT_CLEAN"]
    .apply(scrub_leakage_terms_strong)
)

# --------------------------------------------------
# Clustering-specific administrative cleanup
# Keep separate from supervised no-leak field
# --------------------------------------------------
admin_phrases = [
    # vaccine / manufacturer boilerplate
    r"\bcovid vaccine\b",
    r"\bcovid immunisation\b",
    r"\bmrna moderna\b",
    r"\bmoderna covid\b",
    r"\bpfizer[-\s]?biontech\b",
    r"\bbiontech\b",
    r"\bmoderna\b",
    r"\bpfizer\b",
    r"\bjanssen\b",
    r"\bnovavax\b",
    r"\bbnt\b",
    r"\bmrna\b",
    r"\bvaccine\b",
    r"\bcovid\b",

    # lot / batch / product fields
    r"\blot number\b",
    r"\bbatch number\b",
    r"\bbatch lot\b",
    r"\bvaccine lot\b",
    r"\blot\b",
    r"\bbatch\b",
    r"\bdosage form\b",
    r"\bform\b",
    r"\broute administration\b",
    r"\broute admin\b",
    r"\bunspecified route\b",
    r"\badministration\b",
    r"\bintramuscular\b",
    r"\bsuspension injection\b",
    r"\bsuspension\b",

    # generic case-report boilerplate
    r"\bpatient received\b",
    r"\bpatient experienced\b",
    r"\bsubject experienced\b",
    r"\bspontaneous report\b",
    r"\bmedical history\b",
    r"\bmedical information\b",
    r"\bfollowing information\b",
    r"\bdescribes occurrence\b",
    r"\boccurrence\b",
    r"\bprovided\b",
    r"\bcontactable\b",
    r"\bcase\b",
    r"\breporter\b",
    r"\breported\b",
    r"\breport\b",
    r"\boutcome\b",
    r"\bunknown date\b",
    r"\bdate patient\b",
    r"\bdate\b",

    # dosing / process boilerplate
    r"\bsingle dose\b",
    r"\bdose\b",
    r"\bbooster\b",
    r"\badmin\b",
    r"\bprophylactic vaccination\b",
    r"\breceived\b",
    r"\bgiven\b",
    r"\badministered\b",
    r"\buse\b",

    # recurring vague metadata terms
    r"\bunknown\b",
    r"\binformation\b",
    r"\bdescribed\b",
    r"\bdescribes\b",
    r"\boccurred\b",
    r"\bpatient\b",
    r"\bsubject\b"
]

admin_regexes = [re.compile(p) for p in admin_phrases]

def remove_admin_terms(text):
    if pd.isna(text) or text == "":
        return ""
    text = str(text).lower()
    for rgx in admin_regexes:
        text = rgx.sub(" ", text)
    text = _ws2.sub(" ", text).strip()
    return text

df_clean["SYMPTOM_TEXT_CLEAN_CLUSTER"] = (
    df_clean["SYMPTOM_TEXT_CLEAN"]
    .apply(remove_admin_terms)
)

# --------------------------------------------------
# Optional token count after cluster cleanup
# --------------------------------------------------
df_clean["CLUSTER_TOKEN_COUNT"] = (
    df_clean["SYMPTOM_TEXT_CLEAN_CLUSTER"]
    .str.split()
    .str.len()
    .fillna(0)
    .astype(int)
)

# Optional clustering-specific subset
df_cluster = df_clean[df_clean["CLUSTER_TOKEN_COUNT"] >= 3].copy()

print("Rows in full dataset:", len(df_clean))
print("Rows retained for clustering:", len(df_cluster))
print("Rows dropped as near-empty after cluster cleanup:", len(df_clean) - len(df_cluster))

# --------------------------------------------------
# Save cleaned text variants
# --------------------------------------------------
df_clean[
    [
        "VAERS_ID",
        "SYMPTOM_TEXT",
        "SYMPTOM_TEXT_CLEAN",
        "SYMPTOM_TEXT_CLEAN_NOLEAK",
        "SYMPTOM_TEXT_CLEAN_CLUSTER",
        "CLUSTER_TOKEN_COUNT"
    ]
].to_csv(
    f"{OUTDIR}/symptom_text_clean_variants.csv",
    index=False
)

print([c for c in df_clean.columns if "SYMPTOM" in c or "CLUSTER_TOKEN_COUNT" in c])


In [ ]:
# ============================================================
# 34. TIME SERIES ANALYSIS: MONTHLY REPORTING TRENDS (2020-2025)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

# ------------------------------------------------------------
# 1. Ensure RECVDATE is datetime
# ------------------------------------------------------------
df_ts = df_clean.copy()

df_ts["RECVDATE"] = pd.to_datetime(df_ts["RECVDATE"], errors="coerce")

# Keep only records with valid RECVDATE
df_ts = df_ts[df_ts["RECVDATE"].notna()].copy()

# Restrict to study period 2020-2025
df_ts = df_ts[
    (df_ts["RECVDATE"] >= "2020-01-01") &
    (df_ts["RECVDATE"] < "2026-01-01")
].copy()

print("Time series dataset shape:", df_ts.shape)

# ------------------------------------------------------------
# 2. Create year-month variable
# ------------------------------------------------------------
df_ts["YEAR_MONTH"] = df_ts["RECVDATE"].dt.to_period("M").astype(str)

# Also keep a sortable datetime version for plotting/regression
df_ts["YEAR_MONTH_DATE"] = df_ts["RECVDATE"].dt.to_period("M").dt.to_timestamp()

# ------------------------------------------------------------
# 3. Monthly reporting volume + severe proportion
# ------------------------------------------------------------
monthly_summary = (
    df_ts.groupby("YEAR_MONTH_DATE")
    .agg(
        MONTHLY_REPORTS=("VAERS_ID", "count"),
        SEVERE_CASES=("SERIOUS", "sum")
    )
    .reset_index()
)

monthly_summary["SEVERE_PROPORTION"] = (
    monthly_summary["SEVERE_CASES"] / monthly_summary["MONTHLY_REPORTS"]
)

# Create numeric time index for regression
monthly_summary = monthly_summary.sort_values("YEAR_MONTH_DATE").reset_index(drop=True)
monthly_summary["MONTH_INDEX"] = np.arange(len(monthly_summary))

print(monthly_summary.head())

# Save summary
monthly_summary.to_csv(f"{OUTDIR}/monthly_reporting_summary_2020_2025.csv", index=False)

In [ ]:
# ============================================================
# 35. LINEAR REGRESSION: MONTHLY REPORTING VOLUME OVER TIME
# ============================================================

# ------------------------------------------------------------
# Model 1: Monthly reporting volume ~ time
# ------------------------------------------------------------
X_volume = sm.add_constant(monthly_summary["MONTH_INDEX"])
y_volume = monthly_summary["MONTHLY_REPORTS"]

volume_model = sm.OLS(y_volume, X_volume).fit(cov_type="HC3")

print("MONTHLY REPORTING VOLUME MODEL")
print(volume_model.summary())

# Save model coefficients
volume_results = pd.DataFrame({
    "term": volume_model.params.index,
    "coefficient": volume_model.params.values,
    "p_value": volume_model.pvalues.values,
    "conf_low": volume_model.conf_int()[0].values,
    "conf_high": volume_model.conf_int()[1].values
})
volume_results.to_csv(f"{OUTDIR}/monthly_volume_regression_results.csv", index=False)

# ------------------------------------------------------------
# Plot monthly volume with fitted regression line
# ------------------------------------------------------------
monthly_summary["VOLUME_PRED"] = volume_model.predict(X_volume)

plt.figure(figsize=(12, 5))
plt.plot(monthly_summary["YEAR_MONTH_DATE"], monthly_summary["MONTHLY_REPORTS"], label="Observed monthly reports")
plt.plot(monthly_summary["YEAR_MONTH_DATE"], monthly_summary["VOLUME_PRED"], label="Linear trend", linewidth=2)
plt.title("Monthly COVID-19 VAERS Reporting Volume (2020-2025)")
plt.xlabel("Month")
plt.ylabel("Number of reports")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 36. LINEAR REGRESSION: MONTHLY SEVERE-EVENT PROPORTION OVER TIME
# ============================================================

# ------------------------------------------------------------
# Model 2: Monthly severe proportion ~ time
# ------------------------------------------------------------
X_severe = sm.add_constant(monthly_summary["MONTH_INDEX"])
y_severe = monthly_summary["SEVERE_PROPORTION"]

severe_model = sm.OLS(y_severe, X_severe).fit(cov_type="HC3")

print("MONTHLY SEVERE PROPORTION MODEL")
print(severe_model.summary())

# Save model coefficients
severe_results = pd.DataFrame({
    "term": severe_model.params.index,
    "coefficient": severe_model.params.values,
    "p_value": severe_model.pvalues.values,
    "conf_low": severe_model.conf_int()[0].values,
    "conf_high": severe_model.conf_int()[1].values
})
severe_results.to_csv(f"{OUTDIR}/monthly_severe_proportion_regression_results.csv", index=False)

# ------------------------------------------------------------
# Plot monthly severe-event proportion with fitted regression line
# ------------------------------------------------------------
monthly_summary["SEVERE_PROP_PRED"] = severe_model.predict(X_severe)

plt.figure(figsize=(12, 5))
plt.plot(monthly_summary["YEAR_MONTH_DATE"], monthly_summary["SEVERE_PROPORTION"], label="Observed severe proportion")
plt.plot(monthly_summary["YEAR_MONTH_DATE"], monthly_summary["SEVERE_PROP_PRED"], label="Linear trend", linewidth=2)
plt.title("Monthly Severe-Event Proportion in COVID-19 VAERS Reports (2020-2025)")
plt.xlabel("Month")
plt.ylabel("Proportion severe")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 37. MONTHLY CLUSTER PREVALENCE OVER TIME
# ============================================================

# Make sure df_clean has CLUSTER column
if "CLUSTER" not in df_clean.columns:
    print("⚠️ CLUSTER column not found in df_clean")
    print("You need to either:")
    print("  1. Run the clustering code (steps 18-21) in this session, OR")
    print("  2. Load a pre-clustered dataset")
else:
    # Start fresh from df_clean
    df_ts = df_clean.copy()
    
    # Ensure RECVDATE is datetime
    df_ts["RECVDATE"] = pd.to_datetime(df_ts["RECVDATE"], errors="coerce")
    
    # Create YEAR_MONTH_DATE BEFORE filtering
    df_ts["YEAR_MONTH_DATE"] = df_ts["RECVDATE"].dt.to_period("M").dt.to_timestamp()
    
    # Now filter to rows with cluster labels
    df_cluster_ts = df_ts[df_ts["CLUSTER"].notna()].copy()
    df_cluster_ts["CLUSTER"] = df_cluster_ts["CLUSTER"].astype(int)
    
    print("Rows with cluster labels:", df_cluster_ts.shape[0])
    print("Unique clusters:", sorted(df_cluster_ts["CLUSTER"].unique()))
    print("Date range:", df_cluster_ts["YEAR_MONTH_DATE"].min(), "to", df_cluster_ts["YEAR_MONTH_DATE"].max())
    
    # Monthly counts by cluster
    cluster_month_counts = (
        df_cluster_ts.groupby(["YEAR_MONTH_DATE", "CLUSTER"])
        .size()
        .reset_index(name="N_CLUSTER_MONTH")
    )
    
    # Total clustered reports per month
    cluster_month_totals = (
        df_cluster_ts.groupby("YEAR_MONTH_DATE")
        .size()
        .reset_index(name="N_TOTAL_MONTH")
    )
    
    # Merge totals and compute proportions
    cluster_month_props = cluster_month_counts.merge(
        cluster_month_totals,
        on="YEAR_MONTH_DATE",
        how="left"
    )
    
    cluster_month_props["CLUSTER_PROPORTION"] = (
        cluster_month_props["N_CLUSTER_MONTH"] / cluster_month_props["N_TOTAL_MONTH"]
    )
    
    # Save
    cluster_month_props.to_csv(f"{OUTDIR}/monthly_cluster_proportions.csv", index=False)
    
    print("\nCluster-month proportions:")
    print(cluster_month_props.head(10))

In [ ]:
# ============================================================
# 38. PLOT CLUSTER PREVALENCE OVER TIME
# ============================================================

plt.figure(figsize=(14, 7))

for cluster_id in sorted(cluster_month_props["CLUSTER"].unique()):
    cluster_subset = cluster_month_props[cluster_month_props["CLUSTER"] == cluster_id]
    plt.plot(
        cluster_subset["YEAR_MONTH_DATE"],
        cluster_subset["CLUSTER_PROPORTION"],
        label=f"Cluster {cluster_id}"
    )

plt.title("Monthly Prevalence of Narrative Clusters (2020-2025)")
plt.xlabel("Month")
plt.ylabel("Proportion of clustered reports")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 39. LINEAR REGRESSION FOR EACH CLUSTER'S MONTHLY PREVALENCE
# ============================================================

cluster_trend_results = []

for cluster_id in sorted(cluster_month_props["CLUSTER"].unique()):
    cluster_subset = (
        cluster_month_props[cluster_month_props["CLUSTER"] == cluster_id]
        .sort_values("YEAR_MONTH_DATE")
        .reset_index(drop=True)
    )

    cluster_subset["MONTH_INDEX"] = np.arange(len(cluster_subset))

    X_cluster = sm.add_constant(cluster_subset["MONTH_INDEX"])
    y_cluster = cluster_subset["CLUSTER_PROPORTION"]

    model = sm.OLS(y_cluster, X_cluster).fit(cov_type="HC3")

    cluster_trend_results.append({
        "CLUSTER": cluster_id,
        "INTERCEPT": model.params["const"],
        "SLOPE": model.params["MONTH_INDEX"],
        "P_VALUE": model.pvalues["MONTH_INDEX"],
        "R_SQUARED": model.rsquared
    })

cluster_trend_results_df = pd.DataFrame(cluster_trend_results).sort_values("P_VALUE")
print(cluster_trend_results_df)

cluster_trend_results_df.to_csv(f"{OUTDIR}/cluster_trend_regression_results.csv", index=False)

In [ ]:
# ============================================================
# 40. OPTIONAL: ANNOTATED PLOTS FOR KEY ROLLOUT PHASES
# ============================================================

# You can customize these dates depending on what phases
# you want to highlight in your report.
rollout_dates = [
    ("2020-12-01", "Initial rollout"),
    ("2021-03-01", "Expanded eligibility"),
    ("2021-09-01", "Booster period"),
    ("2022-01-01", "Omicron wave"),
    ("2023-01-01", "Post-rollout stabilization")
]

# Monthly volume plot with annotations
plt.figure(figsize=(12, 5))
plt.plot(monthly_summary["YEAR_MONTH_DATE"], monthly_summary["MONTHLY_REPORTS"], label="Monthly reports")

for date_str, label in rollout_dates:
    date_val = pd.to_datetime(date_str)
    plt.axvline(date_val, linestyle="--", alpha=0.6)
    plt.text(date_val, monthly_summary["MONTHLY_REPORTS"].max()*0.95, label,
             rotation=90, verticalalignment="top", fontsize=9)

plt.title("Monthly COVID-19 VAERS Reporting Volume with Key Rollout Phases")
plt.xlabel("Month")
plt.ylabel("Number of reports")
plt.tight_layout()
plt.show()